# SciFact Baseline Dense Retrieval

Retrieval-only baseline experiment on SciFact (query -> embedding -> retrieval).

In [1]:
import os

# Must run before importing numpy/torch/sentence-transformers
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

try:
    import torch
    torch.set_num_threads(1)
    torch.set_num_interop_threads(1)
except Exception:
    pass

print('OpenMP/BLAS thread guards enabled.')


OpenMP/BLAS thread guards enabled.


In [2]:
import importlib, inspect
import load_data
load_data = importlib.reload(load_data)
print('load_data module file:', load_data.__file__)
print('load_scifact_data starts at line:', inspect.getsourcelines(load_data.load_scifact_data)[1])
src = inspect.getsource(load_data.load_scifact_data)
print('contains _load_raw_scifact?', '_load_raw_scifact' in src)


/opt/anaconda3/envs/anlp-hw34/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


load_data module file: /Users/mj/Desktop/CMU/2026 Spring/11711 Advanced Natural Language Processing/ANLP-HW34/hydeOnSciFact/load_data.py
load_scifact_data starts at line: 130
contains _load_raw_scifact? False


In [3]:
from pathlib import Path
import json
from tqdm import tqdm

from config import DATASET_SPLIT, PREFER_BEIR, MAX_QUERIES, TOP_KS, EMBED_MODEL
from load_data import load_scifact_data
from embed import Embedder
from retrieve import build_faiss_index, retrieve_top_k
from evaluate import evaluate_run


In [4]:
# Config for this notebook run
SPLIT = DATASET_SPLIT
PREFER_BEIR_LOCAL = PREFER_BEIR
MAX_QUERIES_LOCAL = MAX_QUERIES
TOP_KS_LOCAL = TOP_KS
MAX_K = max(TOP_KS_LOCAL)
OUTPUT_DIR = Path('results/baseline')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

corpus, queries, qrels, data_source = load_scifact_data(
    split=SPLIT,
    prefer_beir=PREFER_BEIR_LOCAL,
    max_queries=MAX_QUERIES_LOCAL,
)

query_ids = list(queries.keys())
corpus_doc_ids = list(corpus.keys())
corpus_texts = [corpus[did] for did in corpus_doc_ids]

print(f'Loaded dataset source: {data_source}')
print(f'Corpus size: {len(corpus_texts)} | Queries: {len(query_ids)}')

embedder = Embedder(model_name=EMBED_MODEL)
corpus_vectors = embedder.encode(corpus_texts)
index = build_faiss_index(corpus_vectors)

retrieval_texts = [queries[qid] for qid in query_ids]
query_vectors = embedder.encode(retrieval_texts)
_, all_indices = retrieve_top_k(index, query_vectors, MAX_K)

per_query_retrieved = {}
per_query_rows = []
for i, qid in enumerate(query_ids):
    retrieved_doc_ids = [corpus_doc_ids[j] for j in all_indices[i].tolist()]
    per_query_retrieved[qid] = retrieved_doc_ids

    row = {
        'mode': 'baseline',
        'query_id': qid,
        'query': queries[qid],
        'hyde_document': None,
        'gold_doc_ids': sorted(qrels[qid]),
        'retrieved_doc_ids': retrieved_doc_ids,
        'hit@1': int(any(d in qrels[qid] for d in retrieved_doc_ids[:1])),
        'hit@5': int(any(d in qrels[qid] for d in retrieved_doc_ids[:5])),
        'hit@10': int(any(d in qrels[qid] for d in retrieved_doc_ids[:10])),
    }
    per_query_rows.append(row)

metrics = evaluate_run(per_query_retrieved, qrels, TOP_KS_LOCAL)

print('\n=== Baseline Summary ===')
for k in ['Recall@1', 'Recall@5', 'Recall@10', 'MRR@10', 'nDCG@10']:
    if k in metrics:
        print(f'{k:<10}: {metrics[k]:.4f}')

metrics_payload = {
    'config': {
        'mode': 'baseline',
        'split': SPLIT,
        'prefer_beir': PREFER_BEIR_LOCAL,
        'data_source': data_source,
        'embed_model': EMBED_MODEL,
        'top_ks': TOP_KS_LOCAL,
        'max_queries': MAX_QUERIES_LOCAL,
    },
    'baseline': metrics,
}

metrics_path = OUTPUT_DIR / 'metrics.json'
with metrics_path.open('w', encoding='utf-8') as f:
    json.dump(metrics_payload, f, indent=2, ensure_ascii=False)

per_query_path = OUTPUT_DIR / 'per_query_results.jsonl'
with per_query_path.open('w', encoding='utf-8') as f:
    for row in per_query_rows:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

print(f'\nSaved metrics: {metrics_path}')
print(f'Saved per-query results: {per_query_path}')


Generating test split: 100%|██████████| 339/339 [00:00<00:00, 118617.59 examples/s]


Loaded dataset source: mteb/scifact (corpus:corpus/corpus, queries:queries/queries, qrels:default/test)
Corpus size: 5183 | Queries: 300

=== Baseline Summary ===
Recall@1  : 0.5757
Recall@10 : 0.8566
MRR@10    : 0.6936
nDCG@10   : 0.7296

Saved metrics: results/baseline/metrics.json
Saved per-query results: results/baseline/per_query_results.jsonl
